# 01 — Nettoyage des données (Data Cleaning)
**Projet : Scoring de Crédit — LendingClub 2007-2018**  
**Membre 1 — Data Engineer & Feature Architect**

### Objectifs de ce notebook
1. Charger les données brutes (accepted + rejected)
2. Filtrer `loan_status` → garder uniquement 'Fully Paid' et 'Charged Off'
3. Supprimer les colonnes de **data leakage** (valeurs connues APRÈS le défaut)
4. Convertir les colonnes texte en valeurs numériques
5. Traiter les valeurs manquantes (IterativeImputer + MissingIndicator)
6. Sauvegarder le dataset propre dans `data/processed/`

---
## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# ── Chemins ──────────────────────────────────────────────
RAW_ACCEPTED  = 'data/raw/accepted_2007_to_2018Q4.csv'
RAW_REJECTED  = 'data/raw/rejected_2007_to_2018Q4.csv'
OUT_DIR       = 'data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

print('✅ Imports OK')
print(f'📂 Dossier de sortie : {OUT_DIR}')

---
## 1. Chargement des données

In [ ]:
# ── Chargement du fichier ACCEPTED ───────────────────────
# low_memory=False évite les warnings de types mixtes sur 150 colonnes
print('⏳ Chargement accepted... (peut prendre 1-2 min sur 2.9M lignes)')
df_acc = pd.read_csv(RAW_ACCEPTED, low_memory=False)

print(f'\n📊 Accepted dataset chargé :')
print(f'   Lignes   : {df_acc.shape[0]:,}')
print(f'   Colonnes : {df_acc.shape[1]}')

In [ ]:
# ── Chargement du fichier REJECTED ───────────────────────
print('⏳ Chargement rejected...')
df_rej = pd.read_csv(RAW_REJECTED, low_memory=False)

print(f'\n📊 Rejected dataset chargé :')
print(f'   Lignes   : {df_rej.shape[0]:,}')
print(f'   Colonnes : {df_rej.shape[1]}')
print(f'\nColonnes rejected : {df_rej.columns.tolist()}')

In [ ]:
# ── Aperçu du dataset accepted ───────────────────────────
df_acc.head(3)

In [ ]:
# ── Distribution initiale de loan_status ─────────────────
print('📊 Distribution de loan_status (avant filtre) :')
print(df_acc['loan_status'].value_counts())

fig, ax = plt.subplots(figsize=(10, 4))
df_acc['loan_status'].value_counts().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Distribution de loan_status — données brutes')
ax.set_xlabel('Nombre de prêts')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/01_loan_status_brut.png', dpi=120)
plt.show()
print('💾 Graphique sauvegardé')

---
## 2. Filtrage de loan_status

On garde **uniquement** les prêts terminés :
- `Fully Paid` → target = **0** (remboursé)
- `Charged Off` → target = **1** (défaut)

On supprime les prêts en cours (`Current`, `Late`, `In Grace Period`) car leur issue finale est inconnue.

In [ ]:
STATUTS_GARDES = ['Fully Paid', 'Charged Off']

n_avant = len(df_acc)
df = df_acc[df_acc['loan_status'].isin(STATUTS_GARDES)].copy()
n_apres = len(df)

print(f'Lignes avant filtre : {n_avant:>10,}')
print(f'Lignes après filtre : {n_apres:>10,}')
print(f'Lignes supprimées   : {n_avant - n_apres:>10,} ({(n_avant-n_apres)/n_avant*100:.1f}%)')

# Encodage binaire de la target
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)

print(f'\n📊 Distribution de la target :')
counts = df['target'].value_counts()
print(f'   0 — Fully Paid   : {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)')
print(f'   1 — Charged Off  : {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2ecc71', '#e74c3c']
df['target'].value_counts().plot(kind='bar', ax=axes[0], color=colors, rot=0)
axes[0].set_title('Distribution de la target (0 vs 1)')
axes[0].set_xticklabels(['Fully Paid (0)', 'Charged Off (1)'])
axes[0].set_ylabel('Nombre de prêts')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,}', (p.get_x()+0.3, p.get_height()+1000))

df['target'].value_counts().plot(kind='pie', ax=axes[1], colors=colors,
                                  labels=['Fully Paid', 'Charged Off'],
                                  autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proportion du déséquilibre')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/02_target_distribution.png', dpi=120)
plt.show()

---
## 3. Suppression des colonnes de Data Leakage ⚠️

**Concept critique :** certaines colonnes contiennent des informations qui n'existent qu'APRÈS que le prêt ait fait défaut. Les inclure donnerait une précision artificielle de ~99% — un modèle inutilisable en production.

**Règle :** on ne garde que les colonnes **connues au moment de l'octroi du prêt**.

In [ ]:
# ── Colonnes de data leakage (post-défaut) ────────────────
LEAKAGE_COLS = [
    # Informations de remboursement (inconnues à l'octroi)
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt',
    'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',
    # Informations post-octroi
    'next_pymnt_d',
    'debt_settlement_flag',
    'settlement_status', 'settlement_date',
    'settlement_amount', 'settlement_percentage',
    'settlement_term',
    'hardship_flag', 'hardship_type',
    'hardship_reason', 'hardship_status',
    'hardship_amount', 'hardship_start_date',
    'hardship_end_date', 'payment_plan_start_date',
    'hardship_loan_status', 'hardship_dpd',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
    # Colonnes inutiles / identifiants
    'id', 'member_id', 'url', 'desc',
    'loan_status',   # remplacée par 'target'
    'funded_amnt_inv', 'policy_code',
]

# Garder seulement celles qui existent dans le dataset
leakage_presentes = [c for c in LEAKAGE_COLS if c in df.columns]
leakage_absentes  = [c for c in LEAKAGE_COLS if c not in df.columns]

print(f'Colonnes leakage définies    : {len(LEAKAGE_COLS)}')
print(f'Colonnes leakage présentes   : {len(leakage_presentes)}')
print(f'Colonnes leakage absentes    : {len(leakage_absentes)}')
if leakage_absentes:
    print(f'   (Absentes : {leakage_absentes})')

In [ ]:
n_cols_avant = df.shape[1]
df = df.drop(columns=leakage_presentes)
n_cols_apres = df.shape[1]

print(f'Colonnes avant suppression leakage : {n_cols_avant}')
print(f'Colonnes après suppression leakage : {n_cols_apres}')
print(f'Colonnes supprimées                : {n_cols_avant - n_cols_apres}')
print(f'\nShape finale : {df.shape}')

---
## 4. Analyse des valeurs manquantes

In [ ]:
# ── Calcul du taux de NaN par colonne ────────────────────
nan_stats = pd.DataFrame({
    'nb_nan'   : df.isnull().sum(),
    'pct_nan'  : df.isnull().sum() / len(df) * 100
}).sort_values('pct_nan', ascending=False)

nan_stats = nan_stats[nan_stats['nb_nan'] > 0]

print(f'Colonnes avec valeurs manquantes : {len(nan_stats)}')
print('\nTop 20 colonnes les plus lacunaires :')
print(nan_stats.head(20).to_string())

In [ ]:
# ── Visualisation des NaN ─────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
top30 = nan_stats.head(30)
colors = ['#e74c3c' if x > 50 else '#e67e22' if x > 20 else '#3498db'
          for x in top30['pct_nan']]
top30['pct_nan'].plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=50, color='red',    linestyle='--', linewidth=1, label='Seuil 50% (suppression)')
ax.axvline(x=20, color='orange', linestyle='--', linewidth=1, label='Seuil 20% (attention)')
ax.set_title('Top 30 colonnes avec valeurs manquantes (%)')
ax.set_xlabel('% de valeurs manquantes')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_missing_values.png', dpi=120)
plt.show()

In [ ]:
# ── Stratégie de traitement des NaN ──────────────────────
SEUIL_SUPPRESSION = 50  # % de NaN au-dessus duquel on supprime la colonne

cols_a_supprimer = nan_stats[nan_stats['pct_nan'] > SEUIL_SUPPRESSION].index.tolist()
cols_a_imputer   = nan_stats[nan_stats['pct_nan'] <= SEUIL_SUPPRESSION].index.tolist()

print(f'Colonnes à supprimer (> {SEUIL_SUPPRESSION}% NaN) : {len(cols_a_supprimer)}')
for c in cols_a_supprimer:
    print(f'   {c:40s} → {nan_stats.loc[c, "pct_nan"]:.1f}%')

print(f'\nColonnes à imputer  (≤ {SEUIL_SUPPRESSION}% NaN) : {len(cols_a_imputer)}')

In [ ]:
# Suppression des colonnes trop lacunaires
n_avant = df.shape[1]
df = df.drop(columns=cols_a_supprimer)
print(f'Colonnes avant : {n_avant} → après : {df.shape[1]}')
print(f'Shape actuelle : {df.shape}')

---
## 5. Conversion des colonnes texte en numériques

Plusieurs colonnes sont stockées comme des chaînes de caractères alors qu'elles représentent des nombres.

In [ ]:
# ── 5.1 term : '36 months' → 36 ──────────────────────────
print('Avant :', df['term'].value_counts().head())
df['term'] = df['term'].str.extract(r'(\d+)').astype(float)
print('Après :', df['term'].value_counts().head())

In [ ]:
# ── 5.2 emp_length : '10+ years' → 10 ────────────────────
print('Avant :', df['emp_length'].value_counts())

def parse_emp_length(s):
    if pd.isna(s): return np.nan
    s = str(s)
    if '10+' in s: return 10.0
    if '< 1' in s: return 0.5
    nums = ''.join(filter(str.isdigit, s))
    return float(nums) if nums else np.nan

df['emp_length'] = df['emp_length'].apply(parse_emp_length)
print('\nAprès :', df['emp_length'].describe())

In [ ]:
# ── 5.3 int_rate / revol_util : '13.5%' → 13.5 ──────────
for col in ['int_rate', 'revol_util']:
    if df[col].dtype == object:
        df[col] = df[col].str.replace('%', '').str.strip().astype(float)
        print(f'{col} converti : {df[col].describe()["mean"]:.2f} (moyenne)')
    else:
        print(f'{col} déjà numérique')

In [ ]:
# ── 5.4 grade : A→1, B→2, ..., G→7 ──────────────────────
if df['grade'].dtype == object:
    grade_map = {'A':1, 'B':2, 'C':3, 'D':4, 'E':5, 'F':6, 'G':7}
    print('Avant :', df['grade'].value_counts().sort_index().to_dict())
    df['grade'] = df['grade'].map(grade_map)
    print('Après :', df['grade'].value_counts().sort_index().to_dict())

In [ ]:
# ── 5.5 sub_grade : A1→1, A2→2, ..., G5→35 ──────────────
if df['sub_grade'].dtype == object:
    grades  = list('ABCDEFG')
    sub_map = {f'{g}{n}': (grades.index(g))*5 + n
               for g in grades for n in range(1, 6)}
    df['sub_grade'] = df['sub_grade'].map(sub_map)
    print('sub_grade converti. Valeurs uniques :', sorted(df['sub_grade'].dropna().unique())[:10], '...')

In [ ]:
# ── 5.6 Colonnes de dates → datetime ─────────────────────
date_cols = ['issue_d', 'earliest_cr_line']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%b-%Y', errors='coerce')
        print(f'{col} → datetime | ex: {df[col].dropna().iloc[0]}')

In [ ]:
# ── 5.7 Résumé des types après conversion ─────────────────
print('Types de données après conversion :')
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f'   {str(dtype):15s} : {count} colonnes')

---
## 6. Imputation des valeurs manquantes restantes

### Méthode utilisée : IterativeImputer (MICE) ★ *— hors cours*

Contrairement à `SimpleImputer` (qui remplace par la médiane/moyenne), `IterativeImputer` modélise chaque feature avec les autres pour imputer de façon multivariée. C'est plus précis car il exploite les corrélations entre variables.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # doit être importé avant
from sklearn.impute import IterativeImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler

# Séparer colonnes numériques et catégorielles
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols  = df.select_dtypes(include=['object']).columns.tolist()
date_cols_present = df.select_dtypes(include=['datetime64']).columns.tolist()

# Retirer 'target' des colonnes à imputer
numeric_cols = [c for c in numeric_cols if c != 'target']

print(f'Colonnes numériques à imputer : {len(numeric_cols)}')
print(f'Colonnes catégorielles        : {len(object_cols)}')
print(f'Colonnes datetime             : {len(date_cols_present)}')

In [ ]:
# ── MissingIndicator ★ : créer des flags binaires ─────────
# Indique pour chaque colonne si la valeur était manquante AVANT imputation
# Ces flags peuvent eux-mêmes être prédictifs !

cols_avec_nan = [c for c in numeric_cols if df[c].isnull().any()]
print(f'Colonnes numériques avec NaN : {len(cols_avec_nan)}')

if len(cols_avec_nan) > 0:
    indicator = MissingIndicator(features='missing-only')
    flags = indicator.fit_transform(df[cols_avec_nan])
    flag_names = [f'{c}_was_nan' for c in np.array(cols_avec_nan)[indicator.features_]]
    df_flags = pd.DataFrame(flags.astype(int), columns=flag_names, index=df.index)
    df = pd.concat([df, df_flags], axis=1)
    print(f'✅ {len(flag_names)} colonnes indicatrices ajoutées')
    print(f'   Exemple : {flag_names[:5]}')

In [ ]:
# ── IterativeImputer sur variables numériques ★ ───────────
# max_iter=10 pour la rapidité, random_state pour reproductibilité
print('⏳ IterativeImputer en cours... (quelques minutes sur grand dataset)')

imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    verbose=0
)

df_num_imputed = pd.DataFrame(
    imputer.fit_transform(df[numeric_cols]),
    columns=numeric_cols,
    index=df.index
)

# Remplacer les colonnes numériques par les valeurs imputées
df[numeric_cols] = df_num_imputed

print('✅ Imputation numérique terminée')
print(f'NaN restants (numériques) : {df[numeric_cols].isnull().sum().sum()}')

In [ ]:
# ── Imputation simple pour variables catégorielles ────────
# Pour les catégorielles : mode (valeur la plus fréquente)
for col in object_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)

print('✅ Imputation catégorielle terminée')
print(f'NaN totaux restants : {df.isnull().sum().sum()}')

---
## 7. Statistiques avant / après nettoyage

In [ ]:
print('=' * 60)
print('    RAPPORT DE NETTOYAGE — STATISTIQUES AVANT / APRÈS')
print('=' * 60)

stats = {
    'Lignes (avant filtre loan_status)' : df_acc.shape[0],
    'Lignes (après filtre)'             : len(df),
    'Colonnes (brutes)'                 : df_acc.shape[1],
    'Colonnes (après nettoyage)'        : df.shape[1],
    'Colonnes leakage supprimées'       : len(leakage_presentes),
    'Colonnes supprimées (>50% NaN)'    : len(cols_a_supprimer),
    'Colonnes indicatrices ajoutées'    : len(flag_names) if len(cols_avec_nan) > 0 else 0,
    'NaN restants'                      : df.isnull().sum().sum(),
    'Target 0 (Fully Paid)'             : (df['target'] == 0).sum(),
    'Target 1 (Charged Off)'            : (df['target'] == 1).sum(),
    'Taux de défaut (%)'                : f"{(df['target'] == 1).mean()*100:.1f}%",
}

for k, v in stats.items():
    print(f'   {k:<45} : {v:>15,}' if isinstance(v, int) else f'   {k:<45} : {v}')

In [ ]:
# ── Visualisation : distributions des variables clés ──────
key_vars = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low']
key_vars = [c for c in key_vars if c in df.columns]

fig, axes = plt.subplots(1, len(key_vars), figsize=(16, 4))
for ax, col in zip(axes, key_vars):
    df[col].hist(ax=ax, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Distributions des variables clés — données nettoyées', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/04_distributions_cles.png', dpi=120)
plt.show()

In [ ]:
# ── Taux de défaut par grade ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
default_by_grade = df.groupby('grade')['target'].mean() * 100
default_by_grade.plot(kind='bar', ax=ax, color='coral', rot=0)
ax.set_title('Taux de défaut par grade (A=1 → G=7)')
ax.set_xlabel('Grade')
ax.set_ylabel('% de défaut')
ax.set_xticklabels([f'{int(x)}/Grade {chr(64+int(x))}' for x in default_by_grade.index], rotation=30)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x()+0.1, p.get_height()+0.3))
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/05_default_by_grade.png', dpi=120)
plt.show()

---
## 8. Sauvegarde

In [ ]:
# ── Sauvegarde du dataset nettoyé ─────────────────────────
# Format parquet → 5-10x plus compact que CSV, lecture ultra-rapide
out_path = f'{OUT_DIR}/lending_club_clean.parquet'
df.to_parquet(out_path, index=False)

# Vérification de lecture
df_check = pd.read_parquet(out_path)

print(f'✅ Dataset nettoyé sauvegardé :')
print(f'   Chemin  : {out_path}')
print(f'   Shape   : {df_check.shape}')
print(f'   Taille  : {os.path.getsize(out_path)/1e6:.1f} MB')

# Aussi en CSV pour compatibilité avec les membres 2 et 3
out_csv = f'{OUT_DIR}/lending_club_clean.csv'
df.to_csv(out_csv, index=False)
print(f'   CSV     : {out_csv} ({os.path.getsize(out_csv)/1e6:.1f} MB)')

In [ ]:
# ── Résumé final des colonnes ─────────────────────────────
print('Colonnes du dataset final :')
print(df.dtypes.to_string())

In [ ]:
print('\n' + '=' * 60)
print('    ✅  01_CLEANING TERMINÉ')
print('=' * 60)
print(f'\nLivrable : {out_path}')
print(f'Shape    : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Taux défaut : {df["target"].mean()*100:.1f}%')
print('\n→ Prochain notebook : 02_feature_engineering.ipynb')